# 00 · Data audit and retrieval baseline
Builds the MS MARCO data (`data.json`), inspects it, checks for leakage between splits, then scores an **off-the-shelf
baseline**: the untrained student (Qwen3-Embedding-0.6B) embedding both queries and passages in its own space (2 GPUs).

Output (`/kaggle/working`, attach it to `01`): `cache/<mode>/data.json`, `runs/<mode>/metrics.json` (row `base_student_own_index`).

In [ ]:
# setup: clone/pull the repo on Kaggle, put it on sys.path, install what's missing (re-run to pull latest code)
import os, sys, importlib.util
REPO_URL = "https://github.com/pyther-hub/hlm-poc.git"
ON_KAGGLE = os.path.exists("/kaggle/working")
REPO = "/kaggle/working/hlm-poc" if ON_KAGGLE else os.path.abspath("..")
if ON_KAGGLE:
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL} {REPO}
    else:
        !git -C {REPO} pull -q
sys.path.insert(0, REPO)

# HF token (higher rate limits, faster downloads). Kaggle: Add-ons -> Secrets -> add "HF_TOKEN", attach to notebook.
# Spawned workers inherit os.environ. Locally: export HF_TOKEN or `huggingface-cli login`.
if ON_KAGGLE and "HF_TOKEN" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print(f"HF_TOKEN secret not found ({type(e).__name__}): downloads stay unauthenticated")

import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.56"):  # Qwen3 + the dtype= kwarg
    !pip install -q -U "transformers>=4.56"
    print("transformers upgraded: restart the kernel, then re-run this cell")
for pkg in ("datasets", "pyarrow", "accelerate"):
    if importlib.util.find_spec(pkg) is None:
        !pip install -q {pkg}

%load_ext autoreload
%autoreload 2

## Control panel

In [ ]:
import os
import torch
from teacher import QUERY_PROMPT

DEBUG = True    # first pass: small data (~10k passages). Set False for the full run.
NPROCS = 2      # one process per T4

# --- paths. On Kaggle, earlier notebooks' outputs are attached as inputs (read only); set the slugs to yours.
#     Off Kaggle everything lives under the repo, so every IN_* is the same dir.
ROOT = "/kaggle/working" if ON_KAGGLE else REPO
MODE = "debug" if DEBUG else "full"
KAGGLE_IN = "/kaggle/input/notebooks/tensorpanda231"
CACHE_DIR = f"{ROOT}/cache/{MODE}"
DATA_PATH = f"{CACHE_DIR}/data.json"
BASE_EMB_DIR = f"{CACHE_DIR}/emb/student_base"   # the untrained student's own index (baseline only)
RUN_DIR = f"{ROOT}/runs/{MODE}"

# --- data (Tevatron/msmarco-passage train split)
DATA = dict(n_rows=210_000,     # first rows of the split (streamed, cached as parquet)
            core_n=100_000,     # seeded sample of queries -> train / val / test (split by query)
            n_val=5_000, n_test=5_000,
            n_align=200_000,    # alignment queries = train + other queries, never val/test
            split_seed=1234,
            use_title=False)
if DEBUG:
    DATA.update(n_rows=12_000, core_n=5_000, n_val=500, n_test=500, n_align=10_000)

# --- baseline: the untrained student embeds corpus + val/test queries itself (teacher.embed_main, 2 GPUs)
BASELINE = dict(model="Qwen/Qwen3-Embedding-0.6B", dim=1024, dtype="fp16",   # full native dim, no MRL cut
                query_prompt=QUERY_PROMPT,                                    # the model's own query instruction
                doc_max_len=128, query_max_len=64, batch_size=128,
                sets=("corpus", "val", "test"), data_path=DATA_PATH, emb_dir=BASE_EMB_DIR)

## 1 · Build data
Rows → split by query (train / val / test from a seeded core sample, the rest "other") → corpus = positives + one
stored negative of every core query, deduplicated → alignment queries = train + other, never val/test.

In [ ]:
from data import build_data, load_json, load_rows, save_json

if os.path.exists(DATA_PATH):
    data = load_json(DATA_PATH)
    print("[data] cached")
else:
    rows = load_rows(DATA["n_rows"], CACHE_DIR)
    data = build_data(rows, **{k: v for k, v in DATA.items() if k != "n_rows"})
    save_json(data, DATA_PATH)
print({k: len(v["ids"]) for k, v in data.items()})

## 2 · Audit
What one training example looks like, positives per query, and text lengths in tokens (the `max_len` settings
truncate above them). `negative` is `negative_passages[0]` of the dataset: the "stored negative". It is **not
verified to be a BM25 negative**.

In [ ]:
import numpy as np
from transformers import AutoTokenizer

corpus = data["corpus"]["texts"]
for i in range(3):
    print(f"query    {data['train']['texts'][i]}")
    print(f"positive {corpus[data['train']['pos'][i][0]][:200]}")
    print(f"negative {corpus[data['train']['neg'][i]][:200]}\n")

n_pos = np.array([len(p) for s in ("train", "val", "test") for p in data[s]["pos"]])
print(f"positives per query: mean {n_pos.mean():.2f}, max {n_pos.max()}, share with >1: {(n_pos > 1).mean():.3f}")

tok = AutoTokenizer.from_pretrained(BASELINE["model"])
for name, texts in (("query", data["train"]["texts"]), ("passage", corpus)):
    lens = np.array([len(x) for x in tok(texts[:2000])["input_ids"]])
    print(f"{name:<8} tokens p50 {np.percentile(lens, 50):.0f}  p95 {np.percentile(lens, 95):.0f}  max {lens.max()}")

## 3 · Leakage checks
Hard failures: a query in two of train/val/test, or a val/test query (by id or by normalised text) among the
alignment queries. Reported only: val/test positives that are also train positives. Passages are shared by design
(one corpus), so this is expected and not a leak of queries.

In [ ]:
ids = {s: set(data[s]["ids"]) for s in ("train", "val", "test", "align")}
text = {s: {t.strip().lower() for t in data[s]["texts"]} for s in ("val", "test", "align")}
assert not ids["train"] & ids["val"] and not ids["train"] & ids["test"] and not ids["val"] & ids["test"], "query in two splits"
assert not ids["align"] & (ids["val"] | ids["test"]), "val/test query id in align"
assert not text["align"] & (text["val"] | text["test"]), "val/test query text in align"
print("no query leakage: splits disjoint, align holds no val/test query (id or text)")

train_pos = {p for ps in data["train"]["pos"] for p in ps}
for s in ("val", "test"):
    shared = sum(any(p in train_pos for p in ps) for ps in data[s]["pos"])
    print(f"{s}: {shared}/{len(data[s]['pos'])} queries have a positive that is also a train positive")

## 4 · Baseline: untrained student on its own index
The student embeds the corpus and the queries itself (both GPUs, one shard each, resumable), then retrieves top 100.
It can't be scored on the teacher index: its vectors live in a different space, so it would score ~0 there.
This row is the "no training at all" reference for `04`.

In [ ]:
from data import load_emb
from ddp import launch
from evaluation import save_metrics, score
from model import get_device
from teacher import embed_main

launch(embed_main, BASELINE, nprocs=NPROCS)

device = get_device()
docs, _ = load_emb(BASE_EMB_DIR, "corpus")
result = {}
for split in ("val", "test"):
    q, _ = load_emb(BASE_EMB_DIR, split)
    result[split] = score(q, docs, data[split]["pos"], 100, device)
    print(split, {k: round(v, 4) for k, v in result[split].items()})
save_metrics(RUN_DIR, {"base_student_own_index": result})